In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [9]:
import os

print("INPUT DIRECTORIES:")
for x in os.listdir("/kaggle/input"):
    print(repr(x))

print("\nFULL TREE:")
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)

In [10]:
from pathlib import Path

DATA_ROOT = Path(
    "/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data"
)

for p in sorted(DATA_ROOT.rglob("*")):
    if p.is_file():
        print(f"{p}  |  {p.stat().st_size / 1024**2:.1f} MB")

In [ ]:
!pip install -q "rapidfuzz>=3.6"

In [11]:
import sys
import pandas
import numpy
import scipy
import sklearn
import lightgbm
import rapidfuzz
import pyarrow
import joblib
import psutil

print("Python:", sys.version)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("lightgbm:", lightgbm.__version__)
print("rapidfuzz:", rapidfuzz.__version__)
print("pyarrow:", pyarrow.__version__)
print("joblib:", joblib.__version__)
print("psutil:", psutil.__version__)

In [13]:
!pip install "rapidfuzz>=3.6"

In [14]:
import rapidfuzz
print("RapidFuzz:", rapidfuzz.__version__)

In [16]:
from pathlib import Path
import shutil

SRC = Path("/kaggle/input/datasets/fuckinggpu/business-entity-resolution-source")
if not SRC.exists():
    SRC = Path("/kaggle/input/business-entity-resolution-source")
WORK = Path("/kaggle/working/ml_amazon")

# Remove only the partially-created code directory
if (WORK / "code").exists():
    shutil.rmtree(WORK / "code")

(WORK / "code").mkdir(parents=True, exist_ok=True)

if (SRC / "code").exists():
    shutil.copytree(SRC / "code", WORK / "code", dirs_exist_ok=True)
elif (SRC / "business_entity_resolution").exists():
    shutil.copytree(SRC / "business_entity_resolution", WORK / "code" / "business_entity_resolution", dirs_exist_ok=True)
else:
    for p in SRC.glob("*"):
        if p.is_dir() and "business_entity_resolution" in p.name:
            shutil.copytree(p, WORK / "code" / "business_entity_resolution", dirs_exist_ok=True)
            break
        elif (p / "src").exists():
            shutil.copytree(p, WORK / "code" / p.name, dirs_exist_ok=True)

print("Source copied successfully.")
print("\nPython files:")

for p in sorted((WORK / "code").rglob("*.py")):
    print(p.relative_to(WORK))


In [17]:
from pathlib import Path
import sys
import os

REPO_ROOT = Path("/kaggle/working/ml_amazon")
DATASET_ROOT = Path(
    "/kaggle/input/datasets/fuckinggpu/amazon-entity-resolution-data"
)

# Create the repository's expected dataset directory
dataset_dir = REPO_ROOT / "dataset"
dataset_dir.mkdir(parents=True, exist_ok=True)

# Link Kaggle's read-only dataset into the repository layout
for split in ["train", "test"]:
    target = dataset_dir / split
    source = DATASET_ROOT / split

    if target.exists() or target.is_symlink():
        if target.is_symlink():
            target.unlink()
        elif target.is_dir():
            import shutil
            shutil.rmtree(target)

    if source.exists():
        target.symlink_to(source, target_is_directory=True)
        print(f"Linked {split}: {target} -> {source}")
    else:
        print(f"WARNING: {split} does not exist in Kaggle dataset")

# Make the src package importable
SRC_ROOT = REPO_ROOT / "code" / "business_entity_resolution" / "src"
sys.path.insert(0, str(SRC_ROOT))

print("\nREPO_ROOT:", REPO_ROOT)
print("SRC_ROOT:", SRC_ROOT)

# Verify package + config
from business_entity_resolution.config import (
    REPO_ROOT as CONFIG_REPO_ROOT,
    TRAIN_DIR,
    TRAIN_FILES,
    TRAIN_GROUND_TRUTH,
)

print("\nConfig REPO_ROOT:", CONFIG_REPO_ROOT)
print("TRAIN_DIR:", TRAIN_DIR)

print("\nTraining files:")
for name, path in TRAIN_FILES.items():
    print(f"{name}: {path} | exists={path.exists()}")

print(
    "GROUND TRUTH:",
    TRAIN_GROUND_TRUTH,
    "| exists=",
    TRAIN_GROUND_TRUTH.exists(),
)

In [18]:
import sys
from pathlib import Path

REPO_ROOT = Path("/kaggle/working/ml_amazon")
SRC_ROOT = REPO_ROOT / "code" / "business_entity_resolution" / "src"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Import every major pipeline component.
from business_entity_resolution.config import DEFAULT_CONFIG
from business_entity_resolution.data.loader import *
from business_entity_resolution.normalization.names import *
from business_entity_resolution.normalization.addresses import *
from business_entity_resolution.blocking.candidate_generator import *
from business_entity_resolution.features.build_feature_matrix import *
from business_entity_resolution.models.train_lgbm import *
from business_entity_resolution.models.threshold import *
from business_entity_resolution.pipeline.train_pipeline import *

print("✅ Pipeline imports successful")
print("Random seed:", DEFAULT_CONFIG.random_seed)
print("Token top-k:", DEFAULT_CONFIG.blocking.token_top_k_per_s1)
print("Max candidates/S1:", DEFAULT_CONFIG.blocking.max_candidates_per_s1)
print("LightGBM estimators:", DEFAULT_CONFIG.model.n_estimators)

In [20]:
import inspect
from business_entity_resolution.pipeline import train_pipeline

print("=== FUNCTIONS ===")
for name, obj in inspect.getmembers(train_pipeline, inspect.isfunction):
    print(name, inspect.signature(obj))

print("\n=== MODULE SOURCE ===")
print(inspect.getsource(train_pipeline))

In [21]:
!python /kaggle/working/ml_amazon/code/business_entity_resolution/exp3_script.py